In [1]:
from pathlib import Path
import pandas as pd

DATA_DIR = Path("Fake Image Detection/relevant")
VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

category_aliases = {
    "سباكه": "سباكة",
    "سباكة": "سباكة",
    "كهربا": "كهرباء",
    "كهرباء": "كهرباء",
    "نجاره": "نجارة",
    "نجارة": "نجارة",
    "نقاشه": "نقاشة",
    "نقاشة": "نقاشة",
}

rows = []

for folder in DATA_DIR.iterdir():
    if not folder.is_dir():
        continue

    raw_name = folder.name.strip()
    category = category_aliases.get(raw_name, raw_name)

    for p in folder.rglob("*"):
        if p.is_file() and p.suffix.lower() in VALID_EXTS:
            rows.append({
                "path": str(p),
                "raw_folder": raw_name,
                "category": category
            })

df_stage1c = pd.DataFrame(rows)

print("Total relevant images:", len(df_stage1c))
print("\nCategory distribution:")
print(df_stage1c["category"].value_counts())

print("\nRaw folder distribution:")
print(df_stage1c["raw_folder"].value_counts())

print("\nPreview:")
print(df_stage1c.head())

expected_categories = {"سباكة", "كهرباء", "نجارة", "نقاشة"}
found_categories = set(df_stage1c["category"].unique())

print("\nExpected categories:", expected_categories)
print("Found categories:", found_categories)

missing = expected_categories - found_categories
extra = found_categories - expected_categories

print("\nMissing:", missing)
print("Extra:", extra)

Total relevant images: 2005

Category distribution:
category
نقاشة     733
كهرباء    478
نجارة     438
سباكة     356
Name: count, dtype: int64

Raw folder distribution:
raw_folder
نقاشه    733
كهربا    478
نجاره    438
سباكه    356
Name: count, dtype: int64

Preview:
                                                path raw_folder category
0  Fake Image Detection/relevant/نقاشه/photo_58_2...      نقاشه    نقاشة
1  Fake Image Detection/relevant/نقاشه/photo_2025...      نقاشه    نقاشة
2  Fake Image Detection/relevant/نقاشه/photo_80_2...      نقاشه    نقاشة
3  Fake Image Detection/relevant/نقاشه/photo_13_2...      نقاشه    نقاشة
4  Fake Image Detection/relevant/نقاشه/photo_51_2...      نقاشه    نقاشة

Expected categories: {'سباكة', 'كهرباء', 'نقاشة', 'نجارة'}
Found categories: {'نقاشة', 'سباكة', 'نجارة', 'كهرباء'}

Missing: set()
Extra: set()


In [2]:
# =========================================================
# Stage 1C V3 Training
# Strong Text-Guided Handcrafted Visual Category Verifiers
#
# Goal:
#   image + text_category
#   ↓
#   VERIFIED / NOT_VERIFIED / UNCERTAIN
#
# Training strategy:
#   4 binary one-vs-rest verifiers:
#   - سباكة vs others
#   - كهرباء vs others
#   - نجارة vs others
#   - نقاشة vs others
#
# Features:
#   Handcrafted only:
#   - HOG
#   - Multi-scale LBP
#   - RGB / HSV / LAB histograms
#   - GLCM texture
#   - edge/sharpness/entropy/global stats
#   - 4x4 patch-level color/texture/edge stats
#   - domain-aware color/texture indicators
#
# Outputs:
#   run_stage1c_v3/artifacts/stage1c_v3_bundle.joblib
#   run_stage1c_v3/reports/stage1c_v3_summary.csv
#   run_stage1c_v3/reports/stage1c_v3_summary.json
# =========================================================

import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image, ImageFile

from scipy.stats import entropy, skew

from skimage.color import rgb2gray, rgb2hsv, rgb2lab
from skimage.transform import resize
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops, canny
from skimage.filters import sobel, laplace

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from xgboost import XGBClassifier


warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True
np.random.seed(42)


# =========================================================
# CONFIG
# =========================================================

DATA_DIR = Path("Fake Image Detection/relevant")
RUN_DIR = Path("run_stage1c_v3")
ARTIFACTS_DIR = RUN_DIR / "artifacts"
REPORTS_DIR = RUN_DIR / "reports"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

IMG_SIZE = (224, 224)
PATCH_GRID = 4

TEST_SIZE = 0.20
RANDOM_STATE = 42

K_BEST = 4500

TARGET_ACCEPT_PRECISION = 0.90
MIN_ACCEPT_RECALL = 0.10

TARGET_REJECT_PRECISION = 0.90

USE_HARD_NEGATIVE_MINING = True
HNM_TOP_NEGATIVE_RATIO = 0.12
HNM_DUPLICATION = 2

CATEGORY_ALIASES = {
    "سباكه": "سباكة",
    "سباكة": "سباكة",
    "كهربا": "كهرباء",
    "كهرباء": "كهرباء",
    "نجاره": "نجارة",
    "نجارة": "نجارة",
    "نقاشه": "نقاشة",
    "نقاشة": "نقاشة",
}

CATEGORIES = ["سباكة", "كهرباء", "نجارة", "نقاشة"]


# =========================================================
# DATA LOADING
# =========================================================

def collect_relevant_images(data_dir):
    rows = []

    for folder in data_dir.iterdir():
        if not folder.is_dir():
            continue

        raw_name = folder.name.strip()
        category = CATEGORY_ALIASES.get(raw_name, raw_name)

        if category not in CATEGORIES:
            print(f"⚠️ Skipping unknown category folder: {raw_name}")
            continue

        for p in folder.rglob("*"):
            if p.is_file() and p.suffix.lower() in VALID_EXTS:
                rows.append({
                    "path": str(p),
                    "raw_folder": raw_name,
                    "category": category
                })

    df = pd.DataFrame(rows)

    if len(df) == 0:
        raise ValueError("No relevant images found.")

    return df


# =========================================================
# FEATURE EXTRACTION
# =========================================================

def safe_hist(vals, bins, range_):
    hist, _ = np.histogram(vals, bins=bins, range=range_, density=True)
    hist = np.nan_to_num(hist, nan=0.0, posinf=0.0, neginf=0.0)
    return hist.astype(np.float32)


def color_ratio_features(img_rgb, hsv):
    """
    Domain-aware color indicators.
    These are not semantic models; just handcrafted visual cues.
    """

    r = img_rgb[:, :, 0]
    g = img_rgb[:, :, 1]
    b = img_rgb[:, :, 2]

    h = hsv[:, :, 0]
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]

    # Approximate color masks
    blue_water = ((h > 0.50) & (h < 0.70) & (s > 0.15) & (v > 0.20)).mean()
    brown_wood = (((h > 0.04) & (h < 0.14) & (s > 0.20)) | ((r > g) & (g > b) & (r > 0.25))).mean()
    white_wall = ((s < 0.18) & (v > 0.55)).mean()
    dark_wire = (v < 0.20).mean()
    high_saturation = (s > 0.55).mean()

    return np.array([
        blue_water,
        brown_wood,
        white_wall,
        dark_wire,
        high_saturation,
        float(np.mean(r)),
        float(np.mean(g)),
        float(np.mean(b)),
        float(np.std(r)),
        float(np.std(g)),
        float(np.std(b)),
    ], dtype=np.float32)


def patch_features(img_rgb, gray, hsv, grid=4):
    """
    4x4 patch-level handcrafted features.
    """

    h, w = gray.shape
    feats = []

    patch_h = h // grid
    patch_w = w // grid

    for i in range(grid):
        for j in range(grid):
            y1 = i * patch_h
            y2 = h if i == grid - 1 else (i + 1) * patch_h
            x1 = j * patch_w
            x2 = w if j == grid - 1 else (j + 1) * patch_w

            p_rgb = img_rgb[y1:y2, x1:x2, :]
            p_gray = gray[y1:y2, x1:x2]
            p_hsv = hsv[y1:y2, x1:x2, :]

            p_edges = canny(p_gray).mean()
            p_sobel = sobel(p_gray).mean()
            p_lap = laplace(p_gray).var()

            gray_hist = safe_hist(p_gray.ravel(), bins=16, range_=(0, 1))
            p_entropy = entropy(gray_hist + 1e-8)

            feats.extend([
                float(p_gray.mean()),
                float(p_gray.std()),
                float(p_hsv[:, :, 1].mean()),
                float(p_hsv[:, :, 2].mean()),
                float(p_edges),
                float(p_sobel),
                float(p_lap),
                float(p_entropy),
            ])

            # RGB means/stds
            for ch in range(3):
                vals = p_rgb[:, :, ch].ravel()
                feats.extend([
                    float(np.mean(vals)),
                    float(np.std(vals)),
                ])

    return np.array(feats, dtype=np.float32)


def extract_stage1c_v3_features(image_path, img_size=IMG_SIZE):
    """
    Strong handcrafted visual features.
    """

    img_pil = Image.open(image_path).convert("RGB")
    orig_w, orig_h = img_pil.size

    img = np.array(img_pil)

    img = resize(
        img,
        img_size,
        anti_aliasing=True,
        preserve_range=True
    ).astype(np.float32)

    if img.max() > 1.0:
        img /= 255.0

    gray = rgb2gray(img)
    hsv = rgb2hsv(img)
    lab = rgb2lab(img)

    features = []

    # -----------------------------------------------------
    # 1) HOG
    # -----------------------------------------------------
    hog_feat = hog(
        gray,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True
    )
    features.extend(hog_feat.astype(np.float32))

    # -----------------------------------------------------
    # 2) Multi-scale LBP
    # -----------------------------------------------------
    lbp_1 = local_binary_pattern(gray, P=8, R=1, method="uniform")
    lbp_1_hist = safe_hist(lbp_1.ravel(), bins=10, range_=(0, 10))

    lbp_2 = local_binary_pattern(gray, P=16, R=2, method="uniform")
    lbp_2_hist = safe_hist(lbp_2.ravel(), bins=18, range_=(0, 18))

    features.extend(lbp_1_hist)
    features.extend(lbp_2_hist)

    # -----------------------------------------------------
    # 3) RGB / HSV / LAB histograms
    # -----------------------------------------------------
    for space in [img, hsv]:
        for ch in range(3):
            features.extend(
                safe_hist(space[:, :, ch].ravel(), bins=32, range_=(0, 1))
            )

    # LAB ranges approximately:
    # L: 0..100, A/B: -128..128
    features.extend(safe_hist(lab[:, :, 0].ravel(), bins=32, range_=(0, 100)))
    features.extend(safe_hist(lab[:, :, 1].ravel(), bins=32, range_=(-128, 128)))
    features.extend(safe_hist(lab[:, :, 2].ravel(), bins=32, range_=(-128, 128)))

    # -----------------------------------------------------
    # 4) Color moments
    # -----------------------------------------------------
    for space in [img, hsv]:
        for ch in range(3):
            vals = space[:, :, ch].ravel()
            features.extend([
                float(np.mean(vals)),
                float(np.std(vals)),
                float(skew(vals)),
            ])

    # LAB moments
    for ch in range(3):
        vals = lab[:, :, ch].ravel()
        features.extend([
            float(np.mean(vals)),
            float(np.std(vals)),
            float(skew(vals)),
        ])

    # -----------------------------------------------------
    # 5) GLCM texture
    # -----------------------------------------------------
    gray_u8 = np.clip(gray * 63, 0, 63).astype(np.uint8)

    glcm = graycomatrix(
        gray_u8,
        distances=[1, 2, 4],
        angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
        levels=64,
        symmetric=True,
        normed=True
    )

    for prop in ["contrast", "dissimilarity", "homogeneity", "energy", "correlation", "ASM"]:
        features.extend(graycoprops(glcm, prop).ravel().astype(np.float32))

    # -----------------------------------------------------
    # 6) Edge / sharpness / entropy / global stats
    # -----------------------------------------------------
    edges = canny(gray)
    edge_density = edges.mean()

    sob = sobel(gray)
    sobel_hist = safe_hist(sob.ravel(), bins=24, range_=(0, 1))

    lap_var = laplace(gray).var()

    gray_hist = safe_hist(gray.ravel(), bins=64, range_=(0, 1))
    ent = entropy(gray_hist + 1e-8)

    features.extend(sobel_hist)

    features.extend([
        float(edge_density),
        float(lap_var),
        float(ent),
        float(orig_w / (orig_h + 1e-8)),
        float(gray.mean()),
        float(gray.std()),
        float(hsv[:, :, 1].mean()),
        float(hsv[:, :, 1].std()),
        float(hsv[:, :, 2].mean()),
        float(hsv[:, :, 2].std()),
        float(orig_w),
        float(orig_h),
    ])

    # -----------------------------------------------------
    # 7) Domain-aware handcrafted indicators
    # -----------------------------------------------------
    features.extend(color_ratio_features(img, hsv))

    # -----------------------------------------------------
    # 8) Patch-level features
    # -----------------------------------------------------
    features.extend(patch_features(img, gray, hsv, grid=PATCH_GRID))

    feat = np.array(features, dtype=np.float32)

    feat = np.nan_to_num(
        feat,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return feat


# =========================================================
# PREPROCESSOR
# =========================================================

class Stage1CV3Preprocessor:
    def __init__(self, k_best=4500):
        self.k_best = k_best
        self.selector = None

    def fit(self, X, y):
        k = min(self.k_best, X.shape[1])
        self.selector = SelectKBest(score_func=f_classif, k=k)
        self.selector.fit(X, y)
        return self

    def transform(self, X):
        return self.selector.transform(X)


# =========================================================
# MODELS
# =========================================================

def build_models(y_train):
    pos = int((y_train == 1).sum())
    neg = int((y_train == 0).sum())

    scale_pos_weight = neg / max(pos, 1)

    xgb = XGBClassifier(
        n_estimators=700,
        max_depth=5,
        learning_rate=0.035,
        subsample=0.90,
        colsample_bytree=0.90,
        gamma=0.15,
        min_child_weight=2,
        reg_alpha=0.15,
        reg_lambda=1.5,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    et = ExtraTreesClassifier(
        n_estimators=700,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    rf = RandomForestClassifier(
        n_estimators=600,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    return {
        "xgb": xgb,
        "extra_trees": et,
        "random_forest": rf
    }


def fit_model_ensemble(X_train, y_train):
    models = build_models(y_train)

    for name, model in models.items():
        print(f"    Training {name}...")
        model.fit(X_train, y_train)

    return models


def ensemble_predict_proba(models, X):
    p_xgb = models["xgb"].predict_proba(X)[:, 1]
    p_et = models["extra_trees"].predict_proba(X)[:, 1]
    p_rf = models["random_forest"].predict_proba(X)[:, 1]

    # Weighted ensemble
    prob = (
        0.45 * p_xgb +
        0.35 * p_et +
        0.20 * p_rf
    )

    return prob


# =========================================================
# THRESHOLDS
# =========================================================

def choose_accept_threshold(y_true, probs, target_precision=0.90, min_recall=0.10):
    thresholds = np.linspace(0.05, 0.95, 181)

    candidates = []

    for t in thresholds:
        pred_accept = probs >= t
        n_accept = int(pred_accept.sum())

        if n_accept == 0:
            continue

        tp = int(((y_true == 1) & pred_accept).sum())
        fp = int(((y_true == 0) & pred_accept).sum())
        fn = int(((y_true == 1) & (~pred_accept)).sum())

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)

        if precision >= target_precision and recall >= min_recall:
            candidates.append((t, precision, recall, n_accept))

    if candidates:
        # lowest threshold satisfying target -> more coverage
        best = sorted(candidates, key=lambda x: x[0])[0]
        return float(best[0]), float(best[1]), float(best[2]), "target_precision"

    # fallback best F1
    best_f1 = -1
    best_tuple = (0.5, 0.0, 0.0)

    for t in thresholds:
        pred = probs >= t
        p = precision_score(y_true, pred, zero_division=0)
        r = recall_score(y_true, pred, zero_division=0)
        f = f1_score(y_true, pred, zero_division=0)

        if f > best_f1:
            best_f1 = f
            best_tuple = (t, p, r)

    return float(best_tuple[0]), float(best_tuple[1]), float(best_tuple[2]), "best_f1_fallback"


def choose_reject_threshold(y_true, probs, target_reject_precision=0.90):
    thresholds = np.linspace(0.05, 0.95, 181)

    candidates = []

    for t in thresholds:
        pred_reject = probs <= t
        n_reject = int(pred_reject.sum())

        if n_reject == 0:
            continue

        tn = int(((y_true == 0) & pred_reject).sum())
        fn = int(((y_true == 1) & pred_reject).sum())

        reject_precision = tn / max(tn + fn, 1)

        if reject_precision >= target_reject_precision:
            candidates.append((t, reject_precision, n_reject))

    if candidates:
        # highest threshold satisfying target -> more rejection coverage
        best = sorted(candidates, key=lambda x: x[0], reverse=True)[0]
        return float(best[0]), float(best[1]), "target_reject_precision"

    # conservative fallback
    neg_probs = probs[y_true == 0]
    if len(neg_probs) == 0:
        return 0.20, 0.0, "fallback"

    t = float(np.quantile(neg_probs, 0.70))
    pred_reject = probs <= t
    tn = int(((y_true == 0) & pred_reject).sum())
    fn = int(((y_true == 1) & pred_reject).sum())
    reject_precision = tn / max(tn + fn, 1)

    return t, float(reject_precision), "fallback_quantile"


def tri_state_metrics(y_true, probs, accept_thr, reject_thr):
    decision = np.full(len(probs), "UNCERTAIN", dtype=object)
    decision[probs >= accept_thr] = "VERIFIED"
    decision[probs <= reject_thr] = "NOT_VERIFIED"

    n_verified = int((decision == "VERIFIED").sum())
    n_not_verified = int((decision == "NOT_VERIFIED").sum())
    n_uncertain = int((decision == "UNCERTAIN").sum())

    verified_precision = int(((decision == "VERIFIED") & (y_true == 1)).sum()) / max(n_verified, 1)
    not_verified_precision = int(((decision == "NOT_VERIFIED") & (y_true == 0)).sum()) / max(n_not_verified, 1)

    coverage = (n_verified + n_not_verified) / len(y_true)

    return {
        "n_verified": n_verified,
        "n_not_verified": n_not_verified,
        "n_uncertain": n_uncertain,
        "verified_precision": verified_precision,
        "not_verified_precision": not_verified_precision,
        "coverage": coverage
    }


# =========================================================
# TRAIN ONE VERIFIER
# =========================================================

def train_one_category_verifier(category, X_train_raw, y_train_cat, X_test_raw, y_test_cat):
    print(f"\n==============================")
    print(f"TRAINING VERIFIER: {category}")
    print(f"==============================")

    preprocessor = Stage1CV3Preprocessor(k_best=K_BEST)
    preprocessor.fit(X_train_raw, y_train_cat)

    X_train = preprocessor.transform(X_train_raw)
    X_test = preprocessor.transform(X_test_raw)

    print("Selected features:", X_train.shape[1])
    print("Train distribution:", np.bincount(y_train_cat))
    print("Test distribution :", np.bincount(y_test_cat))

    # Initial training
    models = fit_model_ensemble(X_train, y_train_cat)

    # Hard negative mining on training set
    if USE_HARD_NEGATIVE_MINING:
        train_probs = ensemble_predict_proba(models, X_train)

        neg_idx = np.where(y_train_cat == 0)[0]
        neg_probs = train_probs[neg_idx]

        if len(neg_idx) > 0:
            n_hard = max(1, int(len(neg_idx) * HNM_TOP_NEGATIVE_RATIO))
            hard_order = np.argsort(neg_probs)[-n_hard:]
            hard_neg_idx = neg_idx[hard_order]

            print(f"Hard negatives found: {len(hard_neg_idx)}")

            X_aug = [X_train]
            y_aug = [y_train_cat]

            for _ in range(HNM_DUPLICATION):
                X_aug.append(X_train[hard_neg_idx])
                y_aug.append(y_train_cat[hard_neg_idx])

            X_train_hnm = np.vstack(X_aug)
            y_train_hnm = np.concatenate(y_aug)

            print("After HNM train distribution:", np.bincount(y_train_hnm))

            models = fit_model_ensemble(X_train_hnm, y_train_hnm)

    test_probs = ensemble_predict_proba(models, X_test)

    accept_thr, accept_prec, accept_rec, accept_mode = choose_accept_threshold(
        y_test_cat,
        test_probs,
        target_precision=TARGET_ACCEPT_PRECISION,
        min_recall=MIN_ACCEPT_RECALL
    )

    reject_thr, reject_prec, reject_mode = choose_reject_threshold(
        y_test_cat,
        test_probs,
        target_reject_precision=TARGET_REJECT_PRECISION
    )

    binary_pred = (test_probs >= 0.50).astype(int)

    acc = accuracy_score(y_test_cat, binary_pred)
    bal_acc = balanced_accuracy_score(y_test_cat, binary_pred)
    precision = precision_score(y_test_cat, binary_pred, zero_division=0)
    recall = recall_score(y_test_cat, binary_pred, zero_division=0)
    f1 = f1_score(y_test_cat, binary_pred, zero_division=0)
    roc_auc = roc_auc_score(y_test_cat, test_probs)
    pr_auc = average_precision_score(y_test_cat, test_probs)

    tri = tri_state_metrics(y_test_cat, test_probs, accept_thr, reject_thr)

    cm = confusion_matrix(y_test_cat, binary_pred)

    print("\n===== HOLDOUT RESULTS =====")
    print("Accuracy        :", round(acc, 4))
    print("Balanced Acc    :", round(bal_acc, 4))
    print("Precision @0.5  :", round(precision, 4))
    print("Recall @0.5     :", round(recall, 4))
    print("F1 @0.5         :", round(f1, 4))
    print("ROC AUC         :", round(roc_auc, 4))
    print("PR AUC          :", round(pr_auc, 4))
    print("Accept threshold:", round(accept_thr, 4), accept_mode)
    print("Accept precision:", round(accept_prec, 4))
    print("Accept recall   :", round(accept_rec, 4))
    print("Reject threshold:", round(reject_thr, 4), reject_mode)
    print("Reject precision:", round(reject_prec, 4))
    print("Tri-state       :", tri)
    print("Confusion matrix:")
    print(cm)

    verifier = {
        "category": category,
        "preprocessor": preprocessor,
        "models": models,
        "ensemble_weights": {
            "xgb": 0.45,
            "extra_trees": 0.35,
            "random_forest": 0.20
        },
        "accept_threshold": accept_thr,
        "reject_threshold": reject_thr,
        "accept_threshold_mode": accept_mode,
        "reject_threshold_mode": reject_mode,
        "selected_feature_count": int(X_train.shape[1])
    }

    report = {
        "category": category,
        "accuracy_050": float(acc),
        "balanced_accuracy_050": float(bal_acc),
        "precision_050": float(precision),
        "recall_050": float(recall),
        "f1_050": float(f1),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "accept_threshold": float(accept_thr),
        "accept_precision": float(accept_prec),
        "accept_recall": float(accept_rec),
        "accept_mode": accept_mode,
        "reject_threshold": float(reject_thr),
        "reject_precision": float(reject_prec),
        "reject_mode": reject_mode,
        "n_verified": tri["n_verified"],
        "n_not_verified": tri["n_not_verified"],
        "n_uncertain": tri["n_uncertain"],
        "verified_precision": float(tri["verified_precision"]),
        "not_verified_precision": float(tri["not_verified_precision"]),
        "coverage": float(tri["coverage"]),
        "confusion_matrix": cm.tolist(),
        "classification_report": classification_report(
            y_test_cat,
            binary_pred,
            digits=4,
            zero_division=0
        )
    }

    return verifier, report


# =========================================================
# MAIN
# =========================================================

def main():
    print("===== STAGE 1C V3 TRAINING START =====")

    df = collect_relevant_images(DATA_DIR)

    print("\nDataset size:", len(df))
    print(df["category"].value_counts())

    # Stable category labels
    df["category"] = df["category"].astype(str)

    print("\nExtracting features...")
    X, kept_paths, kept_categories, failed = [], [], [], []

    for row in tqdm(df.itertuples(index=False), total=len(df)):
        try:
            feat = extract_stage1c_v3_features(row.path)
            X.append(feat)
            kept_paths.append(row.path)
            kept_categories.append(row.category)
        except Exception as e:
            failed.append((row.path, str(e)))

    X = np.asarray(X, dtype=np.float32)
    y_cat = np.asarray(kept_categories)
    paths_arr = np.asarray(kept_paths)

    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    print("\nFeature matrix:", X.shape)
    print("Failed images:", len(failed))

    if failed:
        pd.DataFrame(failed, columns=["path", "error"]).to_csv(
            REPORTS_DIR / "stage1c_v3_failed_images.csv",
            index=False,
            encoding="utf-8-sig"
        )

    # Multi-class stratified holdout split, then train one-vs-rest
    X_train_raw, X_test_raw, y_train_cat, y_test_cat, paths_train, paths_test = train_test_split(
        X,
        y_cat,
        paths_arr,
        test_size=TEST_SIZE,
        stratify=y_cat,
        random_state=RANDOM_STATE
    )

    print("\nHoldout split:")
    print("Train:", X_train_raw.shape)
    print(pd.Series(y_train_cat).value_counts())
    print("\nTest:", X_test_raw.shape)
    print(pd.Series(y_test_cat).value_counts())

    verifiers = {}
    reports = []

    for category in CATEGORIES:
        y_train_binary = (y_train_cat == category).astype(int)
        y_test_binary = (y_test_cat == category).astype(int)

        verifier, report = train_one_category_verifier(
            category=category,
            X_train_raw=X_train_raw,
            y_train_cat=y_train_binary,
            X_test_raw=X_test_raw,
            y_test_cat=y_test_binary
        )

        verifiers[category] = verifier
        reports.append(report)

    summary_df = pd.DataFrame(reports)

    summary_csv_path = REPORTS_DIR / "stage1c_v3_summary.csv"
    summary_json_path = REPORTS_DIR / "stage1c_v3_summary.json"

    summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")

    with open(summary_json_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "task": "text_guided_visual_category_verification_v3",
                "categories": CATEGORIES,
                "img_size": IMG_SIZE,
                "patch_grid": PATCH_GRID,
                "k_best": K_BEST,
                "target_accept_precision": TARGET_ACCEPT_PRECISION,
                "target_reject_precision": TARGET_REJECT_PRECISION,
                "use_hard_negative_mining": USE_HARD_NEGATIVE_MINING,
                "summary": reports
            },
            f,
            ensure_ascii=False,
            indent=2
        )

    bundle = {
        "task": "text_guided_visual_category_verification_v3",
        "version": "stage1c_v3",
        "categories": CATEGORIES,
        "img_size": IMG_SIZE,
        "patch_grid": PATCH_GRID,
        "feature_extractor_name": "stage1c_v3_strong_handcrafted_features",
        "verifiers": verifiers,
        "feature_notes": [
            "HOG",
            "Multi-scale LBP",
            "RGB/HSV/LAB histograms",
            "Color moments",
            "GLCM texture",
            "Edge density",
            "Sobel histogram",
            "Laplacian variance",
            "Entropy",
            "Patch-level 4x4 features",
            "Domain-aware handcrafted color/texture indicators"
        ],
        "summary": reports
    }

    bundle_path = ARTIFACTS_DIR / "stage1c_v3_bundle.joblib"
    joblib.dump(bundle, bundle_path)

    # Save test split paths for later error analysis
    test_split_df = pd.DataFrame({
        "path": paths_test,
        "category": y_test_cat
    })

    test_split_df.to_csv(
        REPORTS_DIR / "stage1c_v3_test_split.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("\n===== STAGE 1C V3 TRAINING FINISHED =====")
    print("Saved bundle:", bundle_path)
    print("Saved summary CSV:", summary_csv_path)
    print("Saved summary JSON:", summary_json_path)

    print("\n===== FINAL SUMMARY =====")
    print(summary_df[
        [
            "category",
            "accuracy_050",
            "balanced_accuracy_050",
            "precision_050",
            "recall_050",
            "f1_050",
            "roc_auc",
            "pr_auc",
            "accept_threshold",
            "accept_precision",
            "accept_recall",
            "reject_threshold",
            "reject_precision",
            "coverage"
        ]
    ])


if __name__ == "__main__":
    main()

===== STAGE 1C V3 TRAINING START =====

Dataset size: 2005
category
نقاشة     733
كهرباء    478
نجارة     438
سباكة     356
Name: count, dtype: int64

Extracting features...


100%|██████████| 2005/2005 [05:02<00:00,  6.62it/s]



Feature matrix: (2005, 6770)
Failed images: 0

Holdout split:
Train: (1604, 6770)
نقاشة     586
كهرباء    382
نجارة     351
سباكة     285
Name: count, dtype: int64

Test: (401, 6770)
نقاشة     147
كهرباء     96
نجارة      87
سباكة      71
Name: count, dtype: int64

TRAINING VERIFIER: سباكة
Selected features: 4500
Train distribution: [1319  285]
Test distribution : [330  71]
    Training xgb...
    Training extra_trees...
    Training random_forest...
Hard negatives found: 158
After HNM train distribution: [1635  285]
    Training xgb...
    Training extra_trees...
    Training random_forest...

===== HOLDOUT RESULTS =====
Accuracy        : 0.8329
Balanced Acc    : 0.5337
Precision @0.5  : 0.8333
Recall @0.5     : 0.0704
F1 @0.5         : 0.1299
ROC AUC         : 0.8376
PR AUC          : 0.557
Accept threshold: 0.455 target_precision
Accept precision: 0.9
Accept recall   : 0.1268
Reject threshold: 0.205 target_reject_precision
Reject precision: 0.9003
Tri-state       : {'n_verified': 1

In [1]:
# =========================================================
# Stage 1C V3.1 Training
# Strong Text-Guided Visual Category Verification
#
# Improvements over V3:
# 1) One-vs-rest verifier per category
# 2) Multiclass image category classifier
# 3) Combined final score:
#       final_score =
#           verifier_weight * verifier_prob
#           +
#           multiclass_weight * multiclass_prob_for_text_category
#
# Goal:
#   image + text_category
#   ↓
#   VERIFIED / NOT_VERIFIED
# =========================================================

import json
import warnings
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm
from PIL import Image, ImageFile

from scipy.stats import entropy, skew

from skimage.color import rgb2gray, rgb2hsv, rgb2lab
from skimage.transform import resize
from skimage.feature import hog, local_binary_pattern, graycomatrix, graycoprops, canny
from skimage.filters import sobel, laplace

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from xgboost import XGBClassifier


warnings.filterwarnings("ignore")
ImageFile.LOAD_TRUNCATED_IMAGES = True
np.random.seed(42)


# =========================================================
# CONFIG
# =========================================================

DATA_DIR = Path("Fake Image Detection/relevant")

RUN_DIR = Path("run_stage1c_v31")
ARTIFACTS_DIR = RUN_DIR / "artifacts"
REPORTS_DIR = RUN_DIR / "reports"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

VALID_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

IMG_SIZE = (224, 224)
PATCH_GRID = 4

TEST_SIZE = 0.20
RANDOM_STATE = 42

K_BEST_VERIFIER = 4500
K_BEST_MULTICLASS = 5000

TARGET_ACCEPT_PRECISION = 0.90
TARGET_REJECT_PRECISION = 0.90
MIN_ACCEPT_RECALL = 0.10

USE_HARD_NEGATIVE_MINING = True
HNM_TOP_NEGATIVE_RATIO = 0.12
HNM_DUPLICATION = 2

VERIFIER_WEIGHT = 0.55
MULTICLASS_WEIGHT = 0.45

CATEGORY_ALIASES = {
    "سباكه": "سباكة",
    "سباكة": "سباكة",
    "كهربا": "كهرباء",
    "كهرباء": "كهرباء",
    "نجاره": "نجارة",
    "نجارة": "نجارة",
    "نقاشه": "نقاشة",
    "نقاشة": "نقاشة",
}

CATEGORIES = ["سباكة", "كهرباء", "نجارة", "نقاشة"]


# =========================================================
# DATA LOADING
# =========================================================

def collect_relevant_images(data_dir):
    rows = []

    for folder in data_dir.iterdir():
        if not folder.is_dir():
            continue

        raw_name = folder.name.strip()
        category = CATEGORY_ALIASES.get(raw_name, raw_name)

        if category not in CATEGORIES:
            print(f"⚠️ Skipping unknown folder: {raw_name}")
            continue

        for p in folder.rglob("*"):
            if p.is_file() and p.suffix.lower() in VALID_EXTS:
                rows.append({
                    "path": str(p),
                    "raw_folder": raw_name,
                    "category": category
                })

    df = pd.DataFrame(rows)

    if len(df) == 0:
        raise ValueError("No images found.")

    return df


# =========================================================
# FEATURE EXTRACTION
# =========================================================

def safe_hist(vals, bins, range_):
    hist, _ = np.histogram(vals, bins=bins, range=range_, density=True)
    hist = np.nan_to_num(hist, nan=0.0, posinf=0.0, neginf=0.0)
    return hist.astype(np.float32)


def color_ratio_features(img_rgb, hsv):
    r = img_rgb[:, :, 0]
    g = img_rgb[:, :, 1]
    b = img_rgb[:, :, 2]

    h = hsv[:, :, 0]
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]

    blue_water = ((h > 0.50) & (h < 0.72) & (s > 0.12) & (v > 0.20)).mean()
    cyan_water = ((h > 0.42) & (h < 0.58) & (s > 0.10) & (v > 0.25)).mean()

    brown_wood = (
        ((h > 0.04) & (h < 0.16) & (s > 0.18))
        |
        ((r > g) & (g > b) & (r > 0.25))
    ).mean()

    white_wall = ((s < 0.18) & (v > 0.55)).mean()
    gray_wall = ((s < 0.12) & (v > 0.25) & (v < 0.80)).mean()

    dark_wire = (v < 0.20).mean()
    black_objects = ((v < 0.15) & (s < 0.60)).mean()

    high_saturation = (s > 0.55).mean()

    return np.array([
        blue_water,
        cyan_water,
        brown_wood,
        white_wall,
        gray_wall,
        dark_wire,
        black_objects,
        high_saturation,
        float(np.mean(r)),
        float(np.mean(g)),
        float(np.mean(b)),
        float(np.std(r)),
        float(np.std(g)),
        float(np.std(b)),
        float(np.mean(s)),
        float(np.std(s)),
        float(np.mean(v)),
        float(np.std(v)),
    ], dtype=np.float32)


def patch_features(img_rgb, gray, hsv, grid=4):
    h, w = gray.shape
    feats = []

    patch_h = h // grid
    patch_w = w // grid

    for i in range(grid):
        for j in range(grid):
            y1 = i * patch_h
            y2 = h if i == grid - 1 else (i + 1) * patch_h

            x1 = j * patch_w
            x2 = w if j == grid - 1 else (j + 1) * patch_w

            p_rgb = img_rgb[y1:y2, x1:x2, :]
            p_gray = gray[y1:y2, x1:x2]
            p_hsv = hsv[y1:y2, x1:x2, :]

            p_edges = canny(p_gray).mean()
            p_sobel = sobel(p_gray).mean()
            p_lap = laplace(p_gray).var()

            gray_hist = safe_hist(p_gray.ravel(), bins=16, range_=(0, 1))
            p_entropy = entropy(gray_hist + 1e-8)

            feats.extend([
                float(p_gray.mean()),
                float(p_gray.std()),
                float(p_hsv[:, :, 0].mean()),
                float(p_hsv[:, :, 1].mean()),
                float(p_hsv[:, :, 2].mean()),
                float(p_edges),
                float(p_sobel),
                float(p_lap),
                float(p_entropy),
            ])

            for ch in range(3):
                vals = p_rgb[:, :, ch].ravel()
                feats.extend([
                    float(np.mean(vals)),
                    float(np.std(vals)),
                ])

    return np.array(feats, dtype=np.float32)


def extract_stage1c_v31_features(image_path, img_size=IMG_SIZE):
    img_pil = Image.open(image_path).convert("RGB")
    orig_w, orig_h = img_pil.size

    img = np.array(img_pil)

    img = resize(
        img,
        img_size,
        anti_aliasing=True,
        preserve_range=True
    ).astype(np.float32)

    if img.max() > 1.0:
        img /= 255.0

    gray = rgb2gray(img)
    hsv = rgb2hsv(img)
    lab = rgb2lab(img)

    features = []

    # HOG
    hog_feat = hog(
        gray,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        feature_vector=True
    )
    features.extend(hog_feat.astype(np.float32))

    # Multi-scale LBP
    lbp_1 = local_binary_pattern(gray, P=8, R=1, method="uniform")
    lbp_1_hist = safe_hist(lbp_1.ravel(), bins=10, range_=(0, 10))

    lbp_2 = local_binary_pattern(gray, P=16, R=2, method="uniform")
    lbp_2_hist = safe_hist(lbp_2.ravel(), bins=18, range_=(0, 18))

    lbp_3 = local_binary_pattern(gray, P=24, R=3, method="uniform")
    lbp_3_hist = safe_hist(lbp_3.ravel(), bins=26, range_=(0, 26))

    features.extend(lbp_1_hist)
    features.extend(lbp_2_hist)
    features.extend(lbp_3_hist)

    # RGB / HSV histograms
    for space in [img, hsv]:
        for ch in range(3):
            features.extend(
                safe_hist(space[:, :, ch].ravel(), bins=32, range_=(0, 1))
            )

    # LAB histograms
    features.extend(safe_hist(lab[:, :, 0].ravel(), bins=32, range_=(0, 100)))
    features.extend(safe_hist(lab[:, :, 1].ravel(), bins=32, range_=(-128, 128)))
    features.extend(safe_hist(lab[:, :, 2].ravel(), bins=32, range_=(-128, 128)))

    # Color moments
    for space in [img, hsv]:
        for ch in range(3):
            vals = space[:, :, ch].ravel()
            features.extend([
                float(np.mean(vals)),
                float(np.std(vals)),
                float(skew(vals)),
            ])

    for ch in range(3):
        vals = lab[:, :, ch].ravel()
        features.extend([
            float(np.mean(vals)),
            float(np.std(vals)),
            float(skew(vals)),
        ])

    # GLCM texture
    gray_u8 = np.clip(gray * 63, 0, 63).astype(np.uint8)

    glcm = graycomatrix(
        gray_u8,
        distances=[1, 2, 4],
        angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
        levels=64,
        symmetric=True,
        normed=True
    )

    for prop in [
        "contrast",
        "dissimilarity",
        "homogeneity",
        "energy",
        "correlation",
        "ASM"
    ]:
        features.extend(graycoprops(glcm, prop).ravel().astype(np.float32))

    # Edge / sharpness / entropy
    edges = canny(gray)
    edge_density = edges.mean()

    sob = sobel(gray)
    sobel_hist = safe_hist(sob.ravel(), bins=24, range_=(0, 1))

    lap_var = laplace(gray).var()

    gray_hist = safe_hist(gray.ravel(), bins=64, range_=(0, 1))
    ent = entropy(gray_hist + 1e-8)

    features.extend(sobel_hist)

    features.extend([
        float(edge_density),
        float(lap_var),
        float(ent),
        float(orig_w / (orig_h + 1e-8)),
        float(gray.mean()),
        float(gray.std()),
        float(hsv[:, :, 0].mean()),
        float(hsv[:, :, 0].std()),
        float(hsv[:, :, 1].mean()),
        float(hsv[:, :, 1].std()),
        float(hsv[:, :, 2].mean()),
        float(hsv[:, :, 2].std()),
        float(orig_w),
        float(orig_h),
    ])

    # Domain indicators
    features.extend(color_ratio_features(img, hsv))

    # Patch features
    features.extend(patch_features(img, gray, hsv, grid=PATCH_GRID))

    feat = np.array(features, dtype=np.float32)

    feat = np.nan_to_num(
        feat,
        nan=0.0,
        posinf=0.0,
        neginf=0.0
    )

    return feat


# =========================================================
# PREPROCESSORS
# =========================================================

class Stage1CV31Preprocessor:
    def __init__(self, k_best=4500):
        self.k_best = k_best
        self.selector = None

    def fit(self, X, y):
        k = min(self.k_best, X.shape[1])
        self.selector = SelectKBest(score_func=f_classif, k=k)
        self.selector.fit(X, y)
        return self

    def transform(self, X):
        return self.selector.transform(X)


# =========================================================
# MODEL BUILDERS
# =========================================================

def build_binary_models(y_train):
    pos = int((y_train == 1).sum())
    neg = int((y_train == 0).sum())
    scale_pos_weight = neg / max(pos, 1)

    xgb = XGBClassifier(
        n_estimators=900,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.90,
        colsample_bytree=0.90,
        gamma=0.15,
        min_child_weight=2,
        reg_alpha=0.15,
        reg_lambda=1.5,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    et = ExtraTreesClassifier(
        n_estimators=900,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    rf = RandomForestClassifier(
        n_estimators=700,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    return {
        "xgb": xgb,
        "extra_trees": et,
        "random_forest": rf
    }


def build_multiclass_models():
    xgb = XGBClassifier(
        n_estimators=900,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.90,
        colsample_bytree=0.90,
        gamma=0.10,
        min_child_weight=2,
        reg_alpha=0.15,
        reg_lambda=1.5,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    et = ExtraTreesClassifier(
        n_estimators=900,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    rf = RandomForestClassifier(
        n_estimators=700,
        max_depth=None,
        min_samples_split=4,
        min_samples_leaf=2,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    return {
        "xgb": xgb,
        "extra_trees": et,
        "random_forest": rf
    }


def fit_model_dict(models, X_train, y_train):
    for name, model in models.items():
        print(f"    Training {name}...")
        model.fit(X_train, y_train)
    return models


def binary_ensemble_proba(models, X):
    p_xgb = models["xgb"].predict_proba(X)[:, 1]
    p_et = models["extra_trees"].predict_proba(X)[:, 1]
    p_rf = models["random_forest"].predict_proba(X)[:, 1]

    return (
        0.45 * p_xgb +
        0.35 * p_et +
        0.20 * p_rf
    )


def multiclass_ensemble_proba(models, X):
    p_xgb = models["xgb"].predict_proba(X)
    p_et = models["extra_trees"].predict_proba(X)
    p_rf = models["random_forest"].predict_proba(X)

    return (
        0.45 * p_xgb +
        0.35 * p_et +
        0.20 * p_rf
    )


# =========================================================
# THRESHOLDS AND METRICS
# =========================================================

def choose_accept_threshold(y_true, probs, target_precision=0.90, min_recall=0.10):
    thresholds = np.linspace(0.05, 0.95, 181)
    candidates = []

    for t in thresholds:
        pred_accept = probs >= t
        n_accept = int(pred_accept.sum())

        if n_accept == 0:
            continue

        tp = int(((y_true == 1) & pred_accept).sum())
        fp = int(((y_true == 0) & pred_accept).sum())
        fn = int(((y_true == 1) & (~pred_accept)).sum())

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)

        if precision >= target_precision and recall >= min_recall:
            candidates.append((t, precision, recall, n_accept))

    if candidates:
        best = sorted(candidates, key=lambda x: x[0])[0]
        return float(best[0]), float(best[1]), float(best[2]), "target_precision"

    best_f1 = -1.0
    best_tuple = (0.5, 0.0, 0.0)

    for t in thresholds:
        pred = probs >= t
        p = precision_score(y_true, pred, zero_division=0)
        r = recall_score(y_true, pred, zero_division=0)
        f = f1_score(y_true, pred, zero_division=0)

        if f > best_f1:
            best_f1 = f
            best_tuple = (t, p, r)

    return float(best_tuple[0]), float(best_tuple[1]), float(best_tuple[2]), "best_f1_fallback"


def choose_reject_threshold(y_true, probs, target_reject_precision=0.90):
    thresholds = np.linspace(0.05, 0.95, 181)
    candidates = []

    for t in thresholds:
        pred_reject = probs <= t
        n_reject = int(pred_reject.sum())

        if n_reject == 0:
            continue

        tn = int(((y_true == 0) & pred_reject).sum())
        fn = int(((y_true == 1) & pred_reject).sum())

        reject_precision = tn / max(tn + fn, 1)

        if reject_precision >= target_reject_precision:
            candidates.append((t, reject_precision, n_reject))

    if candidates:
        best = sorted(candidates, key=lambda x: x[0], reverse=True)[0]
        return float(best[0]), float(best[1]), "target_reject_precision"

    neg_probs = probs[y_true == 0]

    if len(neg_probs) == 0:
        return 0.20, 0.0, "fallback"

    t = float(np.quantile(neg_probs, 0.70))
    pred_reject = probs <= t
    tn = int(((y_true == 0) & pred_reject).sum())
    fn = int(((y_true == 1) & pred_reject).sum())
    reject_precision = tn / max(tn + fn, 1)

    return t, float(reject_precision), "fallback_quantile"


def tri_state_metrics(y_true, probs, accept_thr, reject_thr):
    decision = np.full(len(probs), "UNCERTAIN", dtype=object)
    decision[probs >= accept_thr] = "VERIFIED"
    decision[probs <= reject_thr] = "NOT_VERIFIED"

    n_verified = int((decision == "VERIFIED").sum())
    n_not_verified = int((decision == "NOT_VERIFIED").sum())
    n_uncertain = int((decision == "UNCERTAIN").sum())

    verified_precision = int(((decision == "VERIFIED") & (y_true == 1)).sum()) / max(n_verified, 1)
    not_verified_precision = int(((decision == "NOT_VERIFIED") & (y_true == 0)).sum()) / max(n_not_verified, 1)

    coverage = (n_verified + n_not_verified) / len(y_true)

    return {
        "n_verified": n_verified,
        "n_not_verified": n_not_verified,
        "n_uncertain": n_uncertain,
        "verified_precision": verified_precision,
        "not_verified_precision": not_verified_precision,
        "coverage": coverage
    }


# =========================================================
# TRAIN MULTICLASS
# =========================================================

def train_multiclass_classifier(X_train_raw, y_train_cat, X_test_raw, y_test_cat):
    print("\n==============================")
    print("TRAINING MULTICLASS IMAGE CATEGORY CLASSIFIER")
    print("==============================")

    label_encoder = LabelEncoder()
    y_train_enc = label_encoder.fit_transform(y_train_cat)
    y_test_enc = label_encoder.transform(y_test_cat)

    preprocessor = Stage1CV31Preprocessor(k_best=K_BEST_MULTICLASS)
    preprocessor.fit(X_train_raw, y_train_enc)

    X_train = preprocessor.transform(X_train_raw)
    X_test = preprocessor.transform(X_test_raw)

    models = build_multiclass_models()
    fit_model_dict(models, X_train, y_train_enc)

    probs = multiclass_ensemble_proba(models, X_test)
    pred_enc = np.argmax(probs, axis=1)
    pred_cat = label_encoder.inverse_transform(pred_enc)

    acc = accuracy_score(y_test_cat, pred_cat)
    bal_acc = balanced_accuracy_score(y_test_cat, pred_cat)

    print("\n===== MULTICLASS RESULTS =====")
    print("Accuracy        :", round(acc, 4))
    print("Balanced Acc    :", round(bal_acc, 4))
    print("Classes         :", list(label_encoder.classes_))
    print("Confusion matrix:")
    print(confusion_matrix(y_test_cat, pred_cat, labels=list(label_encoder.classes_)))
    print(classification_report(y_test_cat, pred_cat, digits=4, zero_division=0))

    report = {
        "multiclass_accuracy": float(acc),
        "multiclass_balanced_accuracy": float(bal_acc),
        "classes": list(label_encoder.classes_),
        "classification_report": classification_report(
            y_test_cat,
            pred_cat,
            digits=4,
            zero_division=0
        ),
        "confusion_matrix": confusion_matrix(
            y_test_cat,
            pred_cat,
            labels=list(label_encoder.classes_)
        ).tolist()
    }

    classifier = {
        "preprocessor": preprocessor,
        "models": models,
        "label_encoder": label_encoder,
        "classes": list(label_encoder.classes_),
        "ensemble_weights": {
            "xgb": 0.45,
            "extra_trees": 0.35,
            "random_forest": 0.20
        }
    }

    return classifier, report


# =========================================================
# TRAIN ONE BINARY VERIFIER
# =========================================================

def train_one_verifier(
    category,
    X_train_raw,
    y_train_cat,
    X_test_raw,
    y_test_cat,
    multiclass_classifier
):
    print("\n==============================")
    print(f"TRAINING VERIFIER: {category}")
    print("==============================")

    y_train_binary = (y_train_cat == category).astype(int)
    y_test_binary = (y_test_cat == category).astype(int)

    preprocessor = Stage1CV31Preprocessor(k_best=K_BEST_VERIFIER)
    preprocessor.fit(X_train_raw, y_train_binary)

    X_train = preprocessor.transform(X_train_raw)
    X_test = preprocessor.transform(X_test_raw)

    print("Selected features:", X_train.shape[1])
    print("Train distribution:", np.bincount(y_train_binary))
    print("Test distribution :", np.bincount(y_test_binary))

    models = build_binary_models(y_train_binary)
    fit_model_dict(models, X_train, y_train_binary)

    if USE_HARD_NEGATIVE_MINING:
        train_probs = binary_ensemble_proba(models, X_train)

        neg_idx = np.where(y_train_binary == 0)[0]
        neg_probs = train_probs[neg_idx]

        if len(neg_idx) > 0:
            n_hard = max(1, int(len(neg_idx) * HNM_TOP_NEGATIVE_RATIO))
            hard_order = np.argsort(neg_probs)[-n_hard:]
            hard_neg_idx = neg_idx[hard_order]

            print("Hard negatives found:", len(hard_neg_idx))

            X_aug = [X_train]
            y_aug = [y_train_binary]

            for _ in range(HNM_DUPLICATION):
                X_aug.append(X_train[hard_neg_idx])
                y_aug.append(y_train_binary[hard_neg_idx])

            X_train_hnm = np.vstack(X_aug)
            y_train_hnm = np.concatenate(y_aug)

            print("After HNM distribution:", np.bincount(y_train_hnm))

            models = build_binary_models(y_train_hnm)
            fit_model_dict(models, X_train_hnm, y_train_hnm)

    verifier_probs = binary_ensemble_proba(models, X_test)

    # Multiclass probability for this category
    multi_pre = multiclass_classifier["preprocessor"]
    multi_models = multiclass_classifier["models"]
    label_encoder = multiclass_classifier["label_encoder"]

    X_test_multi = multi_pre.transform(X_test_raw)
    multi_probs_all = multiclass_ensemble_proba(multi_models, X_test_multi)

    cat_idx = list(label_encoder.classes_).index(category)
    multi_cat_probs = multi_probs_all[:, cat_idx]

    final_probs = (
        VERIFIER_WEIGHT * verifier_probs
        +
        MULTICLASS_WEIGHT * multi_cat_probs
    )

    binary_pred_050 = (final_probs >= 0.50).astype(int)

    acc = accuracy_score(y_test_binary, binary_pred_050)
    bal_acc = balanced_accuracy_score(y_test_binary, binary_pred_050)
    precision = precision_score(y_test_binary, binary_pred_050, zero_division=0)
    recall = recall_score(y_test_binary, binary_pred_050, zero_division=0)
    f1 = f1_score(y_test_binary, binary_pred_050, zero_division=0)
    roc_auc = roc_auc_score(y_test_binary, final_probs)
    pr_auc = average_precision_score(y_test_binary, final_probs)

    accept_thr, accept_prec, accept_rec, accept_mode = choose_accept_threshold(
        y_test_binary,
        final_probs,
        target_precision=TARGET_ACCEPT_PRECISION,
        min_recall=MIN_ACCEPT_RECALL
    )

    reject_thr, reject_prec, reject_mode = choose_reject_threshold(
        y_test_binary,
        final_probs,
        target_reject_precision=TARGET_REJECT_PRECISION
    )

    tri = tri_state_metrics(
        y_test_binary,
        final_probs,
        accept_thr,
        reject_thr
    )

    cm = confusion_matrix(y_test_binary, binary_pred_050)

    print("\n===== COMBINED HOLDOUT RESULTS =====")
    print("Accuracy        :", round(acc, 4))
    print("Balanced Acc    :", round(bal_acc, 4))
    print("Precision @0.5  :", round(precision, 4))
    print("Recall @0.5     :", round(recall, 4))
    print("F1 @0.5         :", round(f1, 4))
    print("ROC AUC         :", round(roc_auc, 4))
    print("PR AUC          :", round(pr_auc, 4))
    print("Accept threshold:", round(accept_thr, 4), accept_mode)
    print("Accept precision:", round(accept_prec, 4))
    print("Accept recall   :", round(accept_rec, 4))
    print("Reject threshold:", round(reject_thr, 4), reject_mode)
    print("Reject precision:", round(reject_prec, 4))
    print("Tri-state       :", tri)
    print("Confusion matrix:")
    print(cm)

    verifier = {
        "category": category,
        "preprocessor": preprocessor,
        "models": models,
        "verifier_weight": VERIFIER_WEIGHT,
        "multiclass_weight": MULTICLASS_WEIGHT,
        "accept_threshold": float(accept_thr),
        "reject_threshold": float(reject_thr),
        "selected_feature_count": int(X_train.shape[1])
    }

    report = {
        "category": category,
        "accuracy_050": float(acc),
        "balanced_accuracy_050": float(bal_acc),
        "precision_050": float(precision),
        "recall_050": float(recall),
        "f1_050": float(f1),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "accept_threshold": float(accept_thr),
        "accept_precision": float(accept_prec),
        "accept_recall": float(accept_rec),
        "accept_mode": accept_mode,
        "reject_threshold": float(reject_thr),
        "reject_precision": float(reject_prec),
        "reject_mode": reject_mode,
        "n_verified": tri["n_verified"],
        "n_not_verified": tri["n_not_verified"],
        "n_uncertain": tri["n_uncertain"],
        "verified_precision": float(tri["verified_precision"]),
        "not_verified_precision": float(tri["not_verified_precision"]),
        "coverage": float(tri["coverage"]),
        "confusion_matrix": cm.tolist(),
        "classification_report": classification_report(
            y_test_binary,
            binary_pred_050,
            digits=4,
            zero_division=0
        )
    }

    return verifier, report


# =========================================================
# MAIN
# =========================================================

def main():
    print("===== STAGE 1C V3.1 TRAINING START =====")

    df = collect_relevant_images(DATA_DIR)

    print("\nDataset size:", len(df))
    print(df["category"].value_counts())

    X = []
    kept_paths = []
    kept_categories = []
    failed = []

    print("\nExtracting features...")

    for row in tqdm(df.itertuples(index=False), total=len(df)):
        try:
            feat = extract_stage1c_v31_features(row.path)
            X.append(feat)
            kept_paths.append(row.path)
            kept_categories.append(row.category)
        except Exception as e:
            failed.append((row.path, str(e)))

    X = np.asarray(X, dtype=np.float32)
    y_cat = np.asarray(kept_categories)
    paths_arr = np.asarray(kept_paths)

    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    print("\nFeature matrix:", X.shape)
    print("Failed images:", len(failed))

    if failed:
        pd.DataFrame(failed, columns=["path", "error"]).to_csv(
            REPORTS_DIR / "stage1c_v31_failed_images.csv",
            index=False,
            encoding="utf-8-sig"
        )

    X_train_raw, X_test_raw, y_train_cat, y_test_cat, paths_train, paths_test = train_test_split(
        X,
        y_cat,
        paths_arr,
        test_size=TEST_SIZE,
        stratify=y_cat,
        random_state=RANDOM_STATE
    )

    print("\nHoldout split:")
    print("Train:", X_train_raw.shape)
    print(pd.Series(y_train_cat).value_counts())
    print("\nTest:", X_test_raw.shape)
    print(pd.Series(y_test_cat).value_counts())

    multiclass_classifier, multiclass_report = train_multiclass_classifier(
        X_train_raw=X_train_raw,
        y_train_cat=y_train_cat,
        X_test_raw=X_test_raw,
        y_test_cat=y_test_cat
    )

    verifiers = {}
    reports = []

    for category in CATEGORIES:
        verifier, report = train_one_verifier(
            category=category,
            X_train_raw=X_train_raw,
            y_train_cat=y_train_cat,
            X_test_raw=X_test_raw,
            y_test_cat=y_test_cat,
            multiclass_classifier=multiclass_classifier
        )

        verifiers[category] = verifier
        reports.append(report)

    summary_df = pd.DataFrame(reports)

    summary_csv_path = REPORTS_DIR / "stage1c_v31_summary.csv"
    summary_json_path = REPORTS_DIR / "stage1c_v31_summary.json"

    summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")

    with open(summary_json_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "task": "text_guided_visual_category_verification_v31",
                "categories": CATEGORIES,
                "img_size": IMG_SIZE,
                "patch_grid": PATCH_GRID,
                "k_best_verifier": K_BEST_VERIFIER,
                "k_best_multiclass": K_BEST_MULTICLASS,
                "target_accept_precision": TARGET_ACCEPT_PRECISION,
                "target_reject_precision": TARGET_REJECT_PRECISION,
                "verifier_weight": VERIFIER_WEIGHT,
                "multiclass_weight": MULTICLASS_WEIGHT,
                "use_hard_negative_mining": USE_HARD_NEGATIVE_MINING,
                "multiclass_report": multiclass_report,
                "summary": reports
            },
            f,
            ensure_ascii=False,
            indent=2
        )

    bundle = {
        "task": "text_guided_visual_category_verification_v31",
        "version": "stage1c_v31",
        "categories": CATEGORIES,
        "img_size": IMG_SIZE,
        "patch_grid": PATCH_GRID,
        "feature_extractor_name": "stage1c_v31_strong_handcrafted_features",
        "multiclass_classifier": multiclass_classifier,
        "verifiers": verifiers,
        "verifier_weight": VERIFIER_WEIGHT,
        "multiclass_weight": MULTICLASS_WEIGHT,
        "feature_notes": [
            "HOG",
            "Multi-scale LBP",
            "RGB/HSV/LAB histograms",
            "Color moments",
            "GLCM texture",
            "Edge density",
            "Sobel histogram",
            "Laplacian variance",
            "Entropy",
            "Patch-level 4x4 features",
            "Domain-aware handcrafted color/texture indicators",
            "Multiclass image category probability",
            "One-vs-rest verifier probability"
        ],
        "summary": reports,
        "multiclass_report": multiclass_report
    }

    bundle_path = ARTIFACTS_DIR / "stage1c_v31_bundle.joblib"
    joblib.dump(bundle, bundle_path)

    pd.DataFrame({
        "path": paths_test,
        "category": y_test_cat
    }).to_csv(
        REPORTS_DIR / "stage1c_v31_test_split.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("\n===== STAGE 1C V3.1 TRAINING FINISHED =====")
    print("Saved bundle:", bundle_path)
    print("Saved summary CSV:", summary_csv_path)
    print("Saved summary JSON:", summary_json_path)

    print("\n===== MULTICLASS SUMMARY =====")
    print("Accuracy:", multiclass_report["multiclass_accuracy"])
    print("Balanced Accuracy:", multiclass_report["multiclass_balanced_accuracy"])

    print("\n===== FINAL VERIFIER SUMMARY =====")
    print(summary_df[
        [
            "category",
            "accuracy_050",
            "balanced_accuracy_050",
            "precision_050",
            "recall_050",
            "f1_050",
            "roc_auc",
            "pr_auc",
            "accept_threshold",
            "accept_precision",
            "accept_recall",
            "reject_threshold",
            "reject_precision",
            "coverage"
        ]
    ])


if __name__ == "__main__":
    main()

===== STAGE 1C V3.1 TRAINING START =====

Dataset size: 2005
category
نقاشة     733
كهرباء    478
نجارة     438
سباكة     356
Name: count, dtype: int64

Extracting features...


100%|██████████| 2005/2005 [05:45<00:00,  5.80it/s]



Feature matrix: (2005, 6821)
Failed images: 0

Holdout split:
Train: (1604, 6821)
نقاشة     586
كهرباء    382
نجارة     351
سباكة     285
Name: count, dtype: int64

Test: (401, 6821)
نقاشة     147
كهرباء     96
نجارة      87
سباكة      71
Name: count, dtype: int64

TRAINING MULTICLASS IMAGE CATEGORY CLASSIFIER
    Training xgb...


sh: line 1: nvidia-smi: command not found


    Training extra_trees...
    Training random_forest...

===== MULTICLASS RESULTS =====
Accuracy        : 0.7082
Balanced Acc    : 0.661
Classes         : [np.str_('سباكة'), np.str_('كهرباء'), np.str_('نجارة'), np.str_('نقاشة')]
Confusion matrix:
[[ 29  12  11  19]
 [  9  75   8   4]
 [  9  10  49  19]
 [  6   3   7 131]]
              precision    recall  f1-score   support

       سباكة     0.5472    0.4085    0.4677        71
      كهرباء     0.7500    0.7812    0.7653        96
       نجارة     0.6533    0.5632    0.6049        87
       نقاشة     0.7572    0.8912    0.8187       147

    accuracy                         0.7082       401
   macro avg     0.6769    0.6610    0.6642       401
weighted avg     0.6958    0.7082    0.6974       401


TRAINING VERIFIER: سباكة
Selected features: 4500
Train distribution: [1319  285]
Test distribution : [330  71]
    Training xgb...
    Training extra_trees...
    Training random_forest...
Hard negatives found: 158
After HNM distribution:

In [2]:
# =========================================================
# Stage 1C V3.2 Pairwise Specialist Training
#
# Goal:
#   Improve Stage 1C visual category verification by adding
#   pairwise specialist classifiers between category pairs.
#
# Requires:
#   train_stage1c_v31.py
#
# Outputs:
#   run_stage1c_v32/artifacts/stage1c_v32_pairwise_bundle.joblib
#   run_stage1c_v32/reports/stage1c_v32_pairwise_summary.csv
#   run_stage1c_v32/reports/stage1c_v32_pairwise_summary.json
# =========================================================

import json
import warnings
from pathlib import Path
from itertools import combinations

import joblib
import numpy as np
import pandas as pd
from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report
)
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier
from xgboost import XGBClassifier

from stage1c_v31_features import (
    collect_relevant_images,
    extract_stage1c_v31_features,
    DATA_DIR,
    CATEGORIES,
    IMG_SIZE,
    PATCH_GRID,
    VALID_EXTS,
    RANDOM_STATE
)


warnings.filterwarnings("ignore")
np.random.seed(42)


# =========================================================
# CONFIG
# =========================================================

RUN_DIR = Path("run_stage1c_v32")
ARTIFACTS_DIR = RUN_DIR / "artifacts"
REPORTS_DIR = RUN_DIR / "reports"

ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

TEST_SIZE = 0.20

K_BEST_PAIRWISE = 5000

TARGET_ACCEPT_PRECISION = 0.90
TARGET_REJECT_PRECISION = 0.90
MIN_ACCEPT_RECALL = 0.20

USE_HARD_NEGATIVE_MINING = True
HNM_TOP_RATIO = 0.15
HNM_DUPLICATION = 2

PAIRWISE_PAIRS = list(combinations(CATEGORIES, 2))


# =========================================================
# PREPROCESSOR
# =========================================================

class Stage1CV32PairwisePreprocessor:
    def __init__(self, k_best=5000):
        self.k_best = k_best
        self.selector = None

    def fit(self, X, y):
        k = min(self.k_best, X.shape[1])
        self.selector = SelectKBest(score_func=f_classif, k=k)
        self.selector.fit(X, y)
        return self

    def transform(self, X):
        return self.selector.transform(X)


# =========================================================
# MODELS
# =========================================================

def build_pairwise_models(y_train):
    pos = int((y_train == 1).sum())
    neg = int((y_train == 0).sum())
    scale_pos_weight = neg / max(pos, 1)

    xgb = XGBClassifier(
        n_estimators=900,
        max_depth=5,
        learning_rate=0.03,
        subsample=0.90,
        colsample_bytree=0.90,
        gamma=0.10,
        min_child_weight=2,
        reg_alpha=0.10,
        reg_lambda=1.4,
        objective="binary:logistic",
        eval_metric="logloss",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    et = ExtraTreesClassifier(
        n_estimators=900,
        max_depth=None,
        min_samples_split=3,
        min_samples_leaf=1,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    rf = RandomForestClassifier(
        n_estimators=700,
        max_depth=None,
        min_samples_split=3,
        min_samples_leaf=1,
        class_weight="balanced_subsample",
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

    return {
        "xgb": xgb,
        "extra_trees": et,
        "random_forest": rf
    }


def fit_models(models, X_train, y_train):
    for name, model in models.items():
        print(f"    Training {name}...")
        model.fit(X_train, y_train)
    return models


def pairwise_ensemble_proba(models, X):
    p_xgb = models["xgb"].predict_proba(X)[:, 1]
    p_et = models["extra_trees"].predict_proba(X)[:, 1]
    p_rf = models["random_forest"].predict_proba(X)[:, 1]

    return (
        0.45 * p_xgb +
        0.35 * p_et +
        0.20 * p_rf
    )


# =========================================================
# THRESHOLDS
# =========================================================

def choose_accept_threshold(y_true, probs, target_precision=0.90, min_recall=0.20):
    thresholds = np.linspace(0.05, 0.95, 181)
    candidates = []

    for t in thresholds:
        pred = probs >= t
        n_pred = int(pred.sum())

        if n_pred == 0:
            continue

        tp = int(((y_true == 1) & pred).sum())
        fp = int(((y_true == 0) & pred).sum())
        fn = int(((y_true == 1) & (~pred)).sum())

        precision = tp / max(tp + fp, 1)
        recall = tp / max(tp + fn, 1)

        if precision >= target_precision and recall >= min_recall:
            candidates.append((t, precision, recall, n_pred))

    if candidates:
        best = sorted(candidates, key=lambda x: x[0])[0]
        return float(best[0]), float(best[1]), float(best[2]), "target_precision"

    best_f1 = -1
    best = (0.5, 0.0, 0.0)

    for t in thresholds:
        pred = probs >= t
        p = precision_score(y_true, pred, zero_division=0)
        r = recall_score(y_true, pred, zero_division=0)
        f = f1_score(y_true, pred, zero_division=0)

        if f > best_f1:
            best_f1 = f
            best = (t, p, r)

    return float(best[0]), float(best[1]), float(best[2]), "best_f1_fallback"


def choose_reject_threshold(y_true, probs, target_reject_precision=0.90):
    thresholds = np.linspace(0.05, 0.95, 181)
    candidates = []

    for t in thresholds:
        pred_reject = probs <= t
        n_reject = int(pred_reject.sum())

        if n_reject == 0:
            continue

        tn = int(((y_true == 0) & pred_reject).sum())
        fn = int(((y_true == 1) & pred_reject).sum())

        reject_precision = tn / max(tn + fn, 1)

        if reject_precision >= target_reject_precision:
            candidates.append((t, reject_precision, n_reject))

    if candidates:
        best = sorted(candidates, key=lambda x: x[0], reverse=True)[0]
        return float(best[0]), float(best[1]), "target_reject_precision"

    neg_probs = probs[y_true == 0]

    if len(neg_probs) == 0:
        return 0.20, 0.0, "fallback"

    t = float(np.quantile(neg_probs, 0.70))

    pred_reject = probs <= t
    tn = int(((y_true == 0) & pred_reject).sum())
    fn = int(((y_true == 1) & pred_reject).sum())
    reject_precision = tn / max(tn + fn, 1)

    return t, float(reject_precision), "fallback_quantile"


def tri_state_metrics(y_true, probs, accept_thr, reject_thr):
    decision = np.full(len(probs), "UNCERTAIN", dtype=object)
    decision[probs >= accept_thr] = "PAIR_A"
    decision[probs <= reject_thr] = "PAIR_B"

    n_a = int((decision == "PAIR_A").sum())
    n_b = int((decision == "PAIR_B").sum())
    n_u = int((decision == "UNCERTAIN").sum())

    a_precision = int(((decision == "PAIR_A") & (y_true == 1)).sum()) / max(n_a, 1)
    b_precision = int(((decision == "PAIR_B") & (y_true == 0)).sum()) / max(n_b, 1)

    coverage = (n_a + n_b) / len(y_true)

    return {
        "n_pair_a": n_a,
        "n_pair_b": n_b,
        "n_uncertain": n_u,
        "pair_a_precision": a_precision,
        "pair_b_precision": b_precision,
        "coverage": coverage
    }


# =========================================================
# TRAIN ONE PAIRWISE SPECIALIST
# =========================================================

def train_pairwise_specialist(cat_a, cat_b, X_train_raw, y_train_cat, X_test_raw, y_test_cat):
    pair_name = f"{cat_a}__vs__{cat_b}"

    print("\n" + "=" * 70)
    print(f"TRAINING PAIRWISE SPECIALIST: {cat_a} vs {cat_b}")
    print("=" * 70)

    train_mask = np.isin(y_train_cat, [cat_a, cat_b])
    test_mask = np.isin(y_test_cat, [cat_a, cat_b])

    X_train_pair_raw = X_train_raw[train_mask]
    y_train_pair_cat = y_train_cat[train_mask]

    X_test_pair_raw = X_test_raw[test_mask]
    y_test_pair_cat = y_test_cat[test_mask]

    y_train = (y_train_pair_cat == cat_a).astype(int)
    y_test = (y_test_pair_cat == cat_a).astype(int)

    print("Train pair distribution:")
    print(pd.Series(y_train_pair_cat).value_counts())

    print("Test pair distribution:")
    print(pd.Series(y_test_pair_cat).value_counts())

    preprocessor = Stage1CV32PairwisePreprocessor(k_best=K_BEST_PAIRWISE)
    preprocessor.fit(X_train_pair_raw, y_train)

    X_train = preprocessor.transform(X_train_pair_raw)
    X_test = preprocessor.transform(X_test_pair_raw)

    print("Selected features:", X_train.shape[1])

    models = build_pairwise_models(y_train)
    fit_models(models, X_train, y_train)

    if USE_HARD_NEGATIVE_MINING:
        train_probs = pairwise_ensemble_proba(models, X_train)

        # hard from both sides: uncertain/confusing near 0.5
        uncertainty = np.abs(train_probs - 0.5)
        n_hard = max(1, int(len(train_probs) * HNM_TOP_RATIO))
        hard_idx = np.argsort(uncertainty)[:n_hard]

        print("Hard pairwise samples found:", len(hard_idx))

        X_aug = [X_train]
        y_aug = [y_train]

        for _ in range(HNM_DUPLICATION):
            X_aug.append(X_train[hard_idx])
            y_aug.append(y_train[hard_idx])

        X_train_hnm = np.vstack(X_aug)
        y_train_hnm = np.concatenate(y_aug)

        print("After HNM distribution:", np.bincount(y_train_hnm))

        models = build_pairwise_models(y_train_hnm)
        fit_models(models, X_train_hnm, y_train_hnm)

    probs = pairwise_ensemble_proba(models, X_test)
    pred_050 = (probs >= 0.5).astype(int)

    acc = accuracy_score(y_test, pred_050)
    bal_acc = balanced_accuracy_score(y_test, pred_050)
    precision = precision_score(y_test, pred_050, zero_division=0)
    recall = recall_score(y_test, pred_050, zero_division=0)
    f1 = f1_score(y_test, pred_050, zero_division=0)
    roc_auc = roc_auc_score(y_test, probs)
    pr_auc = average_precision_score(y_test, probs)

    accept_thr, accept_prec, accept_rec, accept_mode = choose_accept_threshold(
        y_test,
        probs,
        target_precision=TARGET_ACCEPT_PRECISION,
        min_recall=MIN_ACCEPT_RECALL
    )

    reject_thr, reject_prec, reject_mode = choose_reject_threshold(
        y_test,
        probs,
        target_reject_precision=TARGET_REJECT_PRECISION
    )

    tri = tri_state_metrics(y_test, probs, accept_thr, reject_thr)
    cm = confusion_matrix(y_test, pred_050)

    print("\n===== PAIRWISE RESULTS =====")
    print("Pair             :", pair_name)
    print("Accuracy         :", round(acc, 4))
    print("Balanced Accuracy:", round(bal_acc, 4))
    print("Precision @0.5   :", round(precision, 4))
    print("Recall @0.5      :", round(recall, 4))
    print("F1 @0.5          :", round(f1, 4))
    print("ROC AUC          :", round(roc_auc, 4))
    print("PR AUC           :", round(pr_auc, 4))
    print("Accept threshold :", round(accept_thr, 4), accept_mode)
    print("Accept precision :", round(accept_prec, 4))
    print("Accept recall    :", round(accept_rec, 4))
    print("Reject threshold :", round(reject_thr, 4), reject_mode)
    print("Reject precision :", round(reject_prec, 4))
    print("Tri-state        :", tri)
    print("Confusion matrix:")
    print(cm)

    specialist = {
        "pair_name": pair_name,
        "cat_a": cat_a,
        "cat_b": cat_b,
        "positive_category": cat_a,
        "negative_category": cat_b,
        "preprocessor": preprocessor,
        "models": models,
        "accept_threshold": float(accept_thr),
        "reject_threshold": float(reject_thr),
        "selected_feature_count": int(X_train.shape[1])
    }

    report = {
        "pair_name": pair_name,
        "cat_a": cat_a,
        "cat_b": cat_b,
        "positive_category": cat_a,
        "negative_category": cat_b,
        "accuracy_050": float(acc),
        "balanced_accuracy_050": float(bal_acc),
        "precision_050_for_cat_a": float(precision),
        "recall_050_for_cat_a": float(recall),
        "f1_050_for_cat_a": float(f1),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "accept_threshold_for_cat_a": float(accept_thr),
        "accept_precision_for_cat_a": float(accept_prec),
        "accept_recall_for_cat_a": float(accept_rec),
        "reject_threshold_for_cat_b": float(reject_thr),
        "reject_precision_for_cat_b": float(reject_prec),
        "coverage": float(tri["coverage"]),
        "n_pair_a": tri["n_pair_a"],
        "n_pair_b": tri["n_pair_b"],
        "n_uncertain": tri["n_uncertain"],
        "pair_a_precision": float(tri["pair_a_precision"]),
        "pair_b_precision": float(tri["pair_b_precision"]),
        "confusion_matrix": cm.tolist(),
        "classification_report": classification_report(
            y_test,
            pred_050,
            target_names=[cat_b, cat_a],
            digits=4,
            zero_division=0
        )
    }

    return specialist, report


# =========================================================
# MAIN
# =========================================================

def main():
    print("===== STAGE 1C V3.2 PAIRWISE TRAINING START =====")

    df = collect_relevant_images(DATA_DIR)

    print("\nDataset size:", len(df))
    print(df["category"].value_counts())

    X = []
    kept_paths = []
    kept_categories = []
    failed = []

    print("\nExtracting features...")

    for row in tqdm(df.itertuples(index=False), total=len(df)):
        try:
            feat = extract_stage1c_v31_features(row.path)
            X.append(feat)
            kept_paths.append(row.path)
            kept_categories.append(row.category)
        except Exception as e:
            failed.append((row.path, str(e)))

    X = np.asarray(X, dtype=np.float32)
    y_cat = np.asarray(kept_categories)
    paths_arr = np.asarray(kept_paths)

    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    print("\nFeature matrix:", X.shape)
    print("Failed images:", len(failed))

    if failed:
        pd.DataFrame(failed, columns=["path", "error"]).to_csv(
            REPORTS_DIR / "stage1c_v32_failed_images.csv",
            index=False,
            encoding="utf-8-sig"
        )

    X_train_raw, X_test_raw, y_train_cat, y_test_cat, paths_train, paths_test = train_test_split(
        X,
        y_cat,
        paths_arr,
        test_size=TEST_SIZE,
        stratify=y_cat,
        random_state=RANDOM_STATE
    )

    print("\nHoldout split:")
    print("Train:", X_train_raw.shape)
    print(pd.Series(y_train_cat).value_counts())
    print("\nTest:", X_test_raw.shape)
    print(pd.Series(y_test_cat).value_counts())

    pairwise_specialists = {}
    reports = []

    for cat_a, cat_b in PAIRWISE_PAIRS:
        specialist, report = train_pairwise_specialist(
            cat_a=cat_a,
            cat_b=cat_b,
            X_train_raw=X_train_raw,
            y_train_cat=y_train_cat,
            X_test_raw=X_test_raw,
            y_test_cat=y_test_cat
        )

        pairwise_specialists[report["pair_name"]] = specialist
        reports.append(report)

    summary_df = pd.DataFrame(reports)

    summary_csv_path = REPORTS_DIR / "stage1c_v32_pairwise_summary.csv"
    summary_json_path = REPORTS_DIR / "stage1c_v32_pairwise_summary.json"

    summary_df.to_csv(summary_csv_path, index=False, encoding="utf-8-sig")

    with open(summary_json_path, "w", encoding="utf-8") as f:
        json.dump(
            {
                "task": "stage1c_v32_pairwise_specialists",
                "categories": CATEGORIES,
                "pairs": [list(p) for p in PAIRWISE_PAIRS],
                "img_size": IMG_SIZE,
                "patch_grid": PATCH_GRID,
                "k_best_pairwise": K_BEST_PAIRWISE,
                "target_accept_precision": TARGET_ACCEPT_PRECISION,
                "target_reject_precision": TARGET_REJECT_PRECISION,
                "min_accept_recall": MIN_ACCEPT_RECALL,
                "use_hard_negative_mining": USE_HARD_NEGATIVE_MINING,
                "summary": reports
            },
            f,
            ensure_ascii=False,
            indent=2
        )

    bundle = {
        "task": "stage1c_v32_pairwise_specialists",
        "version": "stage1c_v32",
        "categories": CATEGORIES,
        "pairs": [list(p) for p in PAIRWISE_PAIRS],
        "img_size": IMG_SIZE,
        "patch_grid": PATCH_GRID,
        "feature_extractor_name": "stage1c_v31_strong_handcrafted_features",
        "pairwise_specialists": pairwise_specialists,
        "feature_notes": [
            "Pairwise specialists between category pairs",
            "Uses same strong handcrafted features from V3.1",
            "HOG",
            "Multi-scale LBP",
            "RGB/HSV/LAB histograms",
            "GLCM texture",
            "Patch-level features",
            "Domain-aware handcrafted indicators"
        ],
        "summary": reports
    }

    bundle_path = ARTIFACTS_DIR / "stage1c_v32_pairwise_bundle.joblib"
    joblib.dump(bundle, bundle_path)

    pd.DataFrame({
        "path": paths_test,
        "category": y_test_cat
    }).to_csv(
        REPORTS_DIR / "stage1c_v32_test_split.csv",
        index=False,
        encoding="utf-8-sig"
    )

    print("\n===== STAGE 1C V3.2 PAIRWISE TRAINING FINISHED =====")
    print("Saved bundle:", bundle_path)
    print("Saved summary CSV:", summary_csv_path)
    print("Saved summary JSON:", summary_json_path)

    print("\n===== FINAL PAIRWISE SUMMARY =====")
    print(summary_df[
        [
            "pair_name",
            "accuracy_050",
            "balanced_accuracy_050",
            "precision_050_for_cat_a",
            "recall_050_for_cat_a",
            "f1_050_for_cat_a",
            "roc_auc",
            "pr_auc",
            "accept_precision_for_cat_a",
            "reject_precision_for_cat_b",
            "coverage"
        ]
    ])


if __name__ == "__main__":
    main()

===== STAGE 1C V3.2 PAIRWISE TRAINING START =====

Dataset size: 2005
category
نقاشة     733
كهرباء    478
نجارة     438
سباكة     356
Name: count, dtype: int64

Extracting features...


100%|██████████| 2005/2005 [05:48<00:00,  5.75it/s]



Feature matrix: (2005, 6821)
Failed images: 0

Holdout split:
Train: (1604, 6821)
نقاشة     586
كهرباء    382
نجارة     351
سباكة     285
Name: count, dtype: int64

Test: (401, 6821)
نقاشة     147
كهرباء     96
نجارة      87
سباكة      71
Name: count, dtype: int64

TRAINING PAIRWISE SPECIALIST: سباكة vs كهرباء
Train pair distribution:
كهرباء    382
سباكة     285
Name: count, dtype: int64
Test pair distribution:
كهرباء    96
سباكة     71
Name: count, dtype: int64
Selected features: 5000
    Training xgb...
    Training extra_trees...
    Training random_forest...
Hard pairwise samples found: 100
After HNM distribution: [470 397]
    Training xgb...
    Training extra_trees...
    Training random_forest...

===== PAIRWISE RESULTS =====
Pair             : سباكة__vs__كهرباء
Accuracy         : 0.7964
Balanced Accuracy: 0.7862
Precision @0.5   : 0.7846
Recall @0.5      : 0.7183
F1 @0.5          : 0.75
ROC AUC          : 0.8994
PR AUC           : 0.8488
Accept threshold : 0.725 target_precis

In [1]:
# =========================================================
# Stage 1C Production Inference
# Text-Guided Visual Category Verifier
#
# Input:
#   image_path + text_category from Stage 2
#
# Output:
#   VERIFIED / NOT_VERIFIED
#
# Uses:
#   V3.1 general verifiers + multiclass image classifier
#   V3.2 pairwise specialist verifiers
#
# Required files:
#   stage1c_v31_features.py
#   run_stage1c_v31/artifacts/stage1c_v31_bundle.joblib
#   run_stage1c_v32/artifacts/stage1c_v32_pairwise_bundle.joblib
# =========================================================
%%writefile stage1c_visual_verifier.py

import warnings
from pathlib import Path

import joblib
import numpy as np

from sklearn.feature_selection import SelectKBest, f_classif

from stage1c_v31_features import (
    extract_stage1c_v31_features,
    IMG_SIZE,
    PATCH_GRID,
    CATEGORIES
)


warnings.filterwarnings("ignore")


# =========================================================
# Required classes for joblib loading
# These class names must exist before joblib.load()
# =========================================================

class Stage1CV31Preprocessor:
    def __init__(self, k_best=4500):
        self.k_best = k_best
        self.selector = None

    def fit(self, X, y):
        k = min(self.k_best, X.shape[1])
        self.selector = SelectKBest(score_func=f_classif, k=k)
        self.selector.fit(X, y)
        return self

    def transform(self, X):
        return self.selector.transform(X)


class Stage1CV32PairwisePreprocessor:
    def __init__(self, k_best=5000):
        self.k_best = k_best
        self.selector = None

    def fit(self, X, y):
        k = min(self.k_best, X.shape[1])
        self.selector = SelectKBest(score_func=f_classif, k=k)
        self.selector.fit(X, y)
        return self

    def transform(self, X):
        return self.selector.transform(X)


# =========================================================
# Config
# =========================================================

DEFAULT_V31_BUNDLE_PATH = "run_stage1c_v31/artifacts/stage1c_v31_bundle.joblib"
DEFAULT_V32_BUNDLE_PATH = "run_stage1c_v32/artifacts/stage1c_v32_pairwise_bundle.joblib"

VALID_CATEGORIES = ["سباكة", "كهرباء", "نجارة", "نقاشة"]

CATEGORY_ALIASES = {
    "سباكه": "سباكة",
    "سباكة": "سباكة",
    "كهربا": "كهرباء",
    "كهرباء": "كهرباء",
    "نجاره": "نجارة",
    "نجارة": "نجارة",
    "نقاشه": "نقاشة",
    "نقاشة": "نقاشة",
}

# Final Stage 1C decision thresholds
FINAL_VERIFY_THRESHOLD = 0.55
FINAL_STRONG_VERIFY_THRESHOLD = 0.65
FINAL_REJECT_THRESHOLD = 0.42

# Score fusion weights
GENERAL_VERIFIER_WEIGHT = 0.40
MULTICLASS_WEIGHT = 0.25
PAIRWISE_WEIGHT = 0.35


# =========================================================
# Utility
# =========================================================

def normalize_category(category):
    category = str(category).strip()
    category = CATEGORY_ALIASES.get(category, category)

    if category not in VALID_CATEGORIES:
        raise ValueError(
            f"Unknown category: {category}. "
            f"Expected one of: {VALID_CATEGORIES}"
        )

    return category


def safe_predict_binary_proba(models, X, weights=None):
    """
    Predict binary probability using model dict.

    Expected models:
    - xgb
    - extra_trees
    - random_forest
    """

    if weights is None:
        weights = {
            "xgb": 0.45,
            "extra_trees": 0.35,
            "random_forest": 0.20
        }

    probs = []

    for name, weight in weights.items():
        model = models[name]

        if hasattr(model, "predict_proba"):
            p = float(model.predict_proba(X)[0, 1])
        elif hasattr(model, "decision_function"):
            score = float(model.decision_function(X)[0])
            p = float(1.0 / (1.0 + np.exp(-score)))
        else:
            p = float(model.predict(X)[0])

        probs.append(weight * p)

    return float(np.sum(probs))


def safe_predict_multiclass_proba(models, X, weights=None):
    """
    Predict multiclass probability using model dict.
    """

    if weights is None:
        weights = {
            "xgb": 0.45,
            "extra_trees": 0.35,
            "random_forest": 0.20
        }

    final_probs = None

    for name, weight in weights.items():
        model = models[name]

        if not hasattr(model, "predict_proba"):
            raise ValueError(f"Multiclass model {name} has no predict_proba().")

        probs = model.predict_proba(X)

        if final_probs is None:
            final_probs = weight * probs
        else:
            final_probs += weight * probs

    return final_probs[0]


def pair_key(cat_a, cat_b):
    return f"{cat_a}__vs__{cat_b}"


def find_pairwise_specialist(pairwise_bundle, text_category, other_category):
    """
    Find the pairwise model between text_category and other_category.

    The trained specialist always predicts probability of cat_a.
    If text_category is cat_b, we invert probability.
    """

    specialists = pairwise_bundle["pairwise_specialists"]

    key_forward = pair_key(text_category, other_category)
    key_reverse = pair_key(other_category, text_category)

    if key_forward in specialists:
        return specialists[key_forward], "forward"

    if key_reverse in specialists:
        return specialists[key_reverse], "reverse"

    return None, None


# =========================================================
# Load Stage 1C
# =========================================================

def load_stage1c(
    v31_bundle_path=DEFAULT_V31_BUNDLE_PATH,
    v32_bundle_path=DEFAULT_V32_BUNDLE_PATH
):
    """
    Load Stage 1C V3.1 and V3.2 artifacts.
    """

    v31_bundle_path = Path(v31_bundle_path)
    v32_bundle_path = Path(v32_bundle_path)

    if not v31_bundle_path.exists():
        raise FileNotFoundError(f"Missing V3.1 bundle: {v31_bundle_path}")

    if not v32_bundle_path.exists():
        raise FileNotFoundError(f"Missing V3.2 pairwise bundle: {v32_bundle_path}")

    v31_bundle = joblib.load(v31_bundle_path)
    v32_bundle = joblib.load(v32_bundle_path)

    required_v31_keys = [
        "multiclass_classifier",
        "verifiers",
        "categories",
        "img_size",
        "feature_extractor_name"
    ]

    required_v32_keys = [
        "pairwise_specialists",
        "categories",
        "pairs",
        "feature_extractor_name"
    ]

    missing_v31 = [k for k in required_v31_keys if k not in v31_bundle]
    missing_v32 = [k for k in required_v32_keys if k not in v32_bundle]

    if missing_v31:
        raise KeyError(f"Missing keys in V3.1 bundle: {missing_v31}")

    if missing_v32:
        raise KeyError(f"Missing keys in V3.2 bundle: {missing_v32}")

    stage1c = {
        "v31": v31_bundle,
        "v32": v32_bundle,
        "categories": VALID_CATEGORIES
    }

    return stage1c


# =========================================================
# Core Scores
# =========================================================

def predict_general_verifier_score(X_raw, text_category, stage1c):
    """
    V3.1 one-vs-rest verifier score for the requested text category.
    """

    v31 = stage1c["v31"]
    verifier = v31["verifiers"][text_category]

    preprocessor = verifier["preprocessor"]
    models = verifier["models"]

    weights = verifier.get(
        "ensemble_weights",
        {
            "xgb": 0.45,
            "extra_trees": 0.35,
            "random_forest": 0.20
        }
    )

    X = preprocessor.transform(X_raw)

    score = safe_predict_binary_proba(
        models=models,
        X=X,
        weights=weights
    )

    return float(score)


def predict_multiclass_category_score(X_raw, text_category, stage1c):
    """
    V3.1 multiclass probability that image category == text_category.
    """

    v31 = stage1c["v31"]

    classifier = v31["multiclass_classifier"]
    preprocessor = classifier["preprocessor"]
    models = classifier["models"]
    label_encoder = classifier["label_encoder"]

    weights = classifier.get(
        "ensemble_weights",
        {
            "xgb": 0.45,
            "extra_trees": 0.35,
            "random_forest": 0.20
        }
    )

    X = preprocessor.transform(X_raw)

    probs = safe_predict_multiclass_proba(
        models=models,
        X=X,
        weights=weights
    )

    classes = list(label_encoder.classes_)

    if text_category not in classes:
        raise ValueError(
            f"Category {text_category} not found in multiclass classes: {classes}"
        )

    cat_idx = classes.index(text_category)
    category_prob = float(probs[cat_idx])

    top_idx = int(np.argmax(probs))
    top_category = classes[top_idx]
    top_prob = float(probs[top_idx])

    all_probs = {
        str(cls): float(probs[i])
        for i, cls in enumerate(classes)
    }

    return category_prob, top_category, top_prob, all_probs


def predict_pairwise_support_scores(X_raw, text_category, stage1c):
    """
    Run all pairwise specialists involving text_category.

    Returns:
    - mean support probability
    - min support probability
    - vote count
    - detailed pairwise scores
    """

    v32 = stage1c["v32"]

    pairwise_details = []
    support_scores = []
    support_votes = 0

    other_categories = [
        c for c in VALID_CATEGORIES
        if c != text_category
    ]

    for other_category in other_categories:
        specialist, direction = find_pairwise_specialist(
            pairwise_bundle=v32,
            text_category=text_category,
            other_category=other_category
        )

        if specialist is None:
            continue

        preprocessor = specialist["preprocessor"]
        models = specialist["models"]

        X = preprocessor.transform(X_raw)

        prob_cat_a = safe_predict_binary_proba(
            models=models,
            X=X,
            weights={
                "xgb": 0.45,
                "extra_trees": 0.35,
                "random_forest": 0.20
            }
        )

        if direction == "forward":
            support_prob = prob_cat_a
            pair_orientation = f"{text_category}__vs__{other_category}"
        else:
            support_prob = 1.0 - prob_cat_a
            pair_orientation = f"{other_category}__vs__{text_category}"

        support_prob = float(support_prob)

        support_scores.append(support_prob)

        if support_prob >= 0.50:
            support_votes += 1

        pairwise_details.append({
            "pair": pair_orientation,
            "text_category": text_category,
            "other_category": other_category,
            "direction": direction,
            "support_prob": support_prob
        })

    if len(support_scores) == 0:
        return 0.0, 0.0, 0, pairwise_details

    pairwise_mean = float(np.mean(support_scores))
    pairwise_min = float(np.min(support_scores))

    return pairwise_mean, pairwise_min, support_votes, pairwise_details


# =========================================================
# Decision Logic
# =========================================================

def decide_stage1c(
    final_score,
    general_score,
    multiclass_score,
    pairwise_mean,
    pairwise_min,
    pairwise_votes,
    multiclass_top_category,
    text_category
):
    """
    Final binary visual verification decision:
    VERIFIED / NOT_VERIFIED
    """

    # Strong case: all evidence supports the text category
    if (
        final_score >= FINAL_STRONG_VERIFY_THRESHOLD
        and pairwise_votes >= 2
        and general_score >= 0.45
    ):
        return "VERIFIED", "strong_visual_support"

    # Normal verified case
    if (
        final_score >= FINAL_VERIFY_THRESHOLD
        and pairwise_votes >= 2
        and pairwise_mean >= 0.50
    ):
        return "VERIFIED", "verified_by_combined_score"

    # If multiclass agrees strongly and pairwise is not against it
    if (
        multiclass_top_category == text_category
        and multiclass_score >= 0.55
        and pairwise_votes >= 2
        and final_score >= 0.50
    ):
        return "VERIFIED", "verified_by_multiclass_agreement"

    # Clear rejection
    if final_score <= FINAL_REJECT_THRESHOLD:
        return "NOT_VERIFIED", "low_final_score"

    if pairwise_votes <= 1 and pairwise_mean < 0.50:
        return "NOT_VERIFIED", "pairwise_disagreement"

    if multiclass_top_category != text_category and multiclass_score < 0.25:
        return "NOT_VERIFIED", "multiclass_disagreement"

    # Production system is binary; uncertain becomes NOT_VERIFIED
    return "NOT_VERIFIED", "not_enough_visual_evidence"


# =========================================================
# Predict Stage 1C
# =========================================================

def predict_stage1c_visual_category(
    image_path,
    text_category,
    stage1c,
    return_details=True
):
    """
    Verify whether the image visually supports the category predicted by Stage 2.

    Args:
        image_path:
            Path to image.

        text_category:
            Category predicted by Stage 2.
            One of: سباكة / كهرباء / نجارة / نقاشة

        stage1c:
            Loaded Stage 1C object from load_stage1c().

    Returns:
        dict with decision and scores.
    """

    image_path = Path(image_path)
    text_category = normalize_category(text_category)

    if not image_path.exists():
        raise FileNotFoundError(f"Image not found: {image_path}")

    features = extract_stage1c_v31_features(
        image_path=image_path,
        img_size=IMG_SIZE
    )

    X_raw = features.reshape(1, -1)

    general_score = predict_general_verifier_score(
        X_raw=X_raw,
        text_category=text_category,
        stage1c=stage1c
    )

    (
        multiclass_score,
        multiclass_top_category,
        multiclass_top_prob,
        multiclass_all_probs
    ) = predict_multiclass_category_score(
        X_raw=X_raw,
        text_category=text_category,
        stage1c=stage1c
    )

    (
        pairwise_mean,
        pairwise_min,
        pairwise_votes,
        pairwise_details
    ) = predict_pairwise_support_scores(
        X_raw=X_raw,
        text_category=text_category,
        stage1c=stage1c
    )

    final_score = float(
        GENERAL_VERIFIER_WEIGHT * general_score
        +
        MULTICLASS_WEIGHT * multiclass_score
        +
        PAIRWISE_WEIGHT * pairwise_mean
    )

    decision, reason = decide_stage1c(
        final_score=final_score,
        general_score=general_score,
        multiclass_score=multiclass_score,
        pairwise_mean=pairwise_mean,
        pairwise_min=pairwise_min,
        pairwise_votes=pairwise_votes,
        multiclass_top_category=multiclass_top_category,
        text_category=text_category
    )

    result = {
        "stage": "Stage 1C",
        "image_path": str(image_path),
        "text_category": text_category,
        "decision": decision,
        "confidence": final_score,
        "reason": reason,
        "general_score": general_score,
        "multiclass_score": multiclass_score,
        "multiclass_top_category": multiclass_top_category,
        "multiclass_top_probability": multiclass_top_prob,
        "pairwise_mean": pairwise_mean,
        "pairwise_min": pairwise_min,
        "pairwise_votes": int(pairwise_votes),
    }

    if return_details:
        result["multiclass_all_probs"] = multiclass_all_probs
        result["pairwise_details"] = pairwise_details

    return result


# =========================================================
# Auto Test Helpers
# =========================================================

def find_first_image_for_category(category, root_dir="Fake Image Detection/relevant"):
    """
    Find one test image from the requested category folder.
    """

    category = normalize_category(category)
    root_dir = Path(root_dir)

    folder_aliases = {
        "سباكة": ["سباكة", "سباكه"],
        "كهرباء": ["كهرباء", "كهربا"],
        "نجارة": ["نجارة", "نجاره"],
        "نقاشة": ["نقاشة", "نقاشه"],
    }

    valid_exts = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

    for folder_name in folder_aliases[category]:
        folder = root_dir / folder_name

        if folder.exists():
            for p in folder.rglob("*"):
                if p.is_file() and p.suffix.lower() in valid_exts:
                    return p

    return None


def test_stage1c():
    stage1c = load_stage1c()

    print("✅ Stage 1C loaded successfully.")
    print("V3.1 task:", stage1c["v31"].get("task"))
    print("V3.2 task:", stage1c["v32"].get("task"))
    print("Categories:", stage1c["categories"])

    test_cases = []

    for category in VALID_CATEGORIES:
        img = find_first_image_for_category(category)

        if img is not None:
            test_cases.append((img, category))

    # Add deliberate mismatch tests if possible
    if len(test_cases) > 0:
        first_img, true_cat = test_cases[0]
        wrong_cat = [c for c in VALID_CATEGORIES if c != true_cat][0]
        test_cases.append((first_img, wrong_cat))

    for image_path, text_category in test_cases:
        result = predict_stage1c_visual_category(
            image_path=image_path,
            text_category=text_category,
            stage1c=stage1c,
            return_details=True
        )

        print("\n" + "=" * 70)
        print("Image:", result["image_path"])
        print("Text category:", result["text_category"])
        print("Decision:", result["decision"])
        print("Reason:", result["reason"])
        print("Confidence:", round(result["confidence"], 4))
        print("General score:", round(result["general_score"], 4))
        print("Multiclass score:", round(result["multiclass_score"], 4))
        print("Multiclass top:", result["multiclass_top_category"], round(result["multiclass_top_probability"], 4))
        print("Pairwise mean:", round(result["pairwise_mean"], 4))
        print("Pairwise min:", round(result["pairwise_min"], 4))
        print("Pairwise votes:", result["pairwise_votes"])

        print("Pairwise details:")
        for item in result["pairwise_details"]:
            print(
                f"- {item['pair']} | vs {item['other_category']} "
                f"=> support={item['support_prob']:.4f}"
            )


if __name__ == "__main__":
    test_stage1c()

sh: line 1: nvidia-smi: command not found


✅ Stage 1C loaded successfully.
V3.1 task: text_guided_visual_category_verification_v31
V3.2 task: stage1c_v32_pairwise_specialists
Categories: ['سباكة', 'كهرباء', 'نجارة', 'نقاشة']

Image: Fake Image Detection/relevant/سباكه/photo_40_2025-12-01_18-05-45.jpg
Text category: سباكة
Decision: VERIFIED
Reason: strong_visual_support
Confidence: 0.9319
General score: 0.9325
Multiclass score: 0.8796
Multiclass top: سباكة 0.8796
Pairwise mean: 0.9687
Pairwise min: 0.9524
Pairwise votes: 3
Pairwise details:
- سباكة__vs__كهرباء | vs كهرباء => support=0.9629
- سباكة__vs__نجارة | vs نجارة => support=0.9524
- سباكة__vs__نقاشة | vs نقاشة => support=0.9909

Image: Fake Image Detection/relevant/كهربا/photo_17_2025-12-01_15-46-05.jpg
Text category: كهرباء
Decision: VERIFIED
Reason: strong_visual_support
Confidence: 0.6873
General score: 0.6246
Multiclass score: 0.6684
Multiclass top: كهرباء 0.6684
Pairwise mean: 0.7723
Pairwise min: 0.7563
Pairwise votes: 3
Pairwise details:
- سباكة__vs__كهرباء | vs سبا